# Holt-Winters Exponential Smoothing for Energy Forecasting

## Overview

This quantlet demonstrates the **Holt-Winters exponential smoothing** method applied to natural gas price forecasting. We compare three increasingly sophisticated approaches:

1. **Simple Exponential Smoothing (SES)**: Suitable for data without trend or seasonality. Uses only a level component.

2. **Holt's Linear Trend Method**: Extends SES by adding a trend component. Captures upward or downward movements in the data.

3. **Holt-Winters Seasonal Method**: Adds a seasonal component to Holt's method. Captures repeating patterns (e.g., annual cycles in energy demand).

## The Holt-Winters Model

The **additive** Holt-Winters model consists of three smoothing equations:

- **Level**: $\ell_t = \alpha(y_t - s_{t-m}) + (1-\alpha)(\ell_{t-1} + b_{t-1})$
- **Trend**: $b_t = \beta(\ell_t - \ell_{t-1}) + (1-\beta)b_{t-1}$
- **Seasonal**: $s_t = \gamma(y_t - \ell_{t-1} - b_{t-1}) + (1-\gamma)s_{t-m}$

Where:
- $\alpha, \beta, \gamma$ are smoothing parameters (0 < parameter < 1)
- $m$ is the seasonal period (12 for monthly data with annual seasonality)

The forecast equation is: $\hat{y}_{t+h} = \ell_t + hb_t + s_{t+h-m}$

## Educational Focus

Energy prices often exhibit **seasonality** due to:
- Heating demand in winter (natural gas)
- Cooling demand in summer (electricity)
- Agricultural cycles (biofuels)

This notebook demonstrates how **Holt-Winters captures seasonality** that simpler exponential smoothing methods miss.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

## 1. Download and Prepare Natural Gas Data

In [ ]:
# Download Natural Gas Futures data
ticker = "NG=F"
data = yf.download(ticker, start="2010-01-01", end="2024-12-31", progress=False)

# Resample to monthly frequency (using last trading day of each month)
monthly_data = data['Close'].resample('ME').last().dropna()

# Ensure we have a proper datetime index with frequency
monthly_data.index = pd.DatetimeIndex(monthly_data.index, freq='ME')

print(f"Data period: {monthly_data.index[0].strftime('%Y-%m')} to {monthly_data.index[-1].strftime('%Y-%m')}")
print(f"Total observations: {len(monthly_data)}")
monthly_data.head(10)

## 2. Train/Test Split (80/20)

In [ ]:
# Split data: 80% training, 20% testing
split_idx = int(len(monthly_data) * 0.8)
train = monthly_data[:split_idx]
test = monthly_data[split_idx:]

print(f"Training set: {train.index[0].strftime('%Y-%m')} to {train.index[-1].strftime('%Y-%m')} ({len(train)} observations)")
print(f"Test set: {test.index[0].strftime('%Y-%m')} to {test.index[-1].strftime('%Y-%m')} ({len(test)} observations)")

## 3. Fit Exponential Smoothing Models

We fit three models with increasing complexity:
1. **Simple ES**: No trend, no seasonality
2. **Holt**: Additive trend, no seasonality
3. **Holt-Winters**: Additive trend + additive seasonality (period=12)

In [ ]:
# Model 1: Simple Exponential Smoothing
model_ses = SimpleExpSmoothing(train, initialization_method='estimated').fit(optimized=True)
forecast_ses = model_ses.forecast(len(test))
print(f"Simple ES - Alpha: {model_ses.params['smoothing_level']:.4f}")

# Model 2: Holt's Linear Trend Method
model_holt = ExponentialSmoothing(
    train,
    trend='add',
    seasonal=None,
    initialization_method='estimated'
).fit(optimized=True)
forecast_holt = model_holt.forecast(len(test))
print(f"Holt - Alpha: {model_holt.params['smoothing_level']:.4f}, Beta: {model_holt.params['smoothing_trend']:.4f}")

# Model 3: Holt-Winters (Additive Seasonal, period=12)
model_hw = ExponentialSmoothing(
    train,
    trend='add',
    seasonal='add',
    seasonal_periods=12,
    initialization_method='estimated'
).fit(optimized=True)
forecast_hw = model_hw.forecast(len(test))
print(f"Holt-Winters - Alpha: {model_hw.params['smoothing_level']:.4f}, "
      f"Beta: {model_hw.params['smoothing_trend']:.4f}, Gamma: {model_hw.params['smoothing_seasonal']:.4f}")

## 4. Forecast Evaluation: MAE and RMSE

In [ ]:
# Calculate error metrics
def calculate_metrics(actual, forecast, model_name):
    mae = mean_absolute_error(actual, forecast)
    rmse = np.sqrt(mean_squared_error(actual, forecast))
    return {'Model': model_name, 'MAE': mae, 'RMSE': rmse}

metrics = [
    calculate_metrics(test, forecast_ses, 'Simple ES'),
    calculate_metrics(test, forecast_holt, 'Holt (Trend)'),
    calculate_metrics(test, forecast_hw, 'Holt-Winters (Seasonal)')
]

metrics_df = pd.DataFrame(metrics)
metrics_df.set_index('Model', inplace=True)
print("\n" + "="*50)
print("Forecast Accuracy Comparison")
print("="*50)
print(metrics_df.round(4).to_string())
print("="*50)

## 5. Visualization: Actual vs Forecasts with Confidence Intervals

In [ ]:
# Create figure with professional styling
fig, ax = plt.subplots(figsize=(14, 7))

# Set transparent background
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

# Plot training data
ax.plot(train.index, train.values, color='#1f77b4', linewidth=1.5, label='Training Data', alpha=0.7)

# Plot actual test data
ax.plot(test.index, test.values, color='black', linewidth=2, label='Actual (Test)', marker='o', markersize=4)

# Plot forecasts
ax.plot(test.index, forecast_ses.values, color='#ff7f0e', linewidth=2, linestyle='--', label='Simple ES', alpha=0.8)
ax.plot(test.index, forecast_holt.values, color='#2ca02c', linewidth=2, linestyle='-.', label='Holt (Trend)', alpha=0.8)
ax.plot(test.index, forecast_hw.values, color='#d62728', linewidth=2.5, label='Holt-Winters (Seasonal)', alpha=0.9)

# Add confidence interval for Holt-Winters (approximate: +/- 1.96 * residual std)
# Use SSE from model to compute residual standard deviation
sse = model_hw.sse
n_fitted = len(model_hw.fittedvalues)
std_resid = np.sqrt(sse / n_fitted)
ci_lower = forecast_hw.values - 1.96 * std_resid
ci_upper = forecast_hw.values + 1.96 * std_resid

ax.fill_between(test.index, ci_lower, ci_upper, color='#d62728', alpha=0.15, label='95% CI (Holt-Winters)')

# Add vertical line to separate train/test
ax.axvline(x=train.index[-1], color='gray', linestyle=':', alpha=0.7, linewidth=1.5)
ax.text(train.index[-1], ax.get_ylim()[1]*0.95, ' Train/Test Split', fontsize=10, color='gray', va='top')

# Remove grid
ax.grid(False)

# Labels and title
ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Natural Gas Price (USD/MMBtu)', fontsize=12, fontweight='bold')
ax.set_title('Natural Gas Price Forecasting: Comparing Exponential Smoothing Methods', 
             fontsize=14, fontweight='bold', pad=20)

# Legend outside the plot
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), frameon=False, ncol=3, fontsize=9)

# Tight layout to accommodate legend
plt.tight_layout()

fig.savefig('holt_winters_forecast.pdf', bbox_inches='tight', transparent=True, dpi=300, facecolor='none', edgecolor='none')
fig.savefig('holt_winters_forecast.png', bbox_inches='tight', transparent=True, dpi=300, facecolor='none', edgecolor='none')
print("Saved: holt_winters_forecast.pdf")
print("Saved: holt_winters_forecast.png")
plt.show()

## 6. Component Decomposition (Holt-Winters)

In [ ]:
# Extract and plot Holt-Winters components
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Set transparent background
fig.patch.set_alpha(0)
for ax in axes:
    ax.patch.set_alpha(0)
    ax.grid(False)

# Get fitted values index (may be shorter due to seasonal initialization)
fitted_idx = model_hw.fittedvalues.index

# Level component
axes[0].plot(fitted_idx, model_hw.level, color='#1f77b4', linewidth=2)
axes[0].set_ylabel('Level', fontsize=11, fontweight='bold')
axes[0].set_title('Holt-Winters Component Decomposition', fontsize=14, fontweight='bold', pad=15)

# Trend component
axes[1].plot(fitted_idx, model_hw.trend, color='#2ca02c', linewidth=2)
axes[1].set_ylabel('Trend', fontsize=11, fontweight='bold')
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Seasonal component
axes[2].plot(fitted_idx, model_hw.season, color='#d62728', linewidth=2)
axes[2].set_ylabel('Seasonal', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Date', fontsize=11, fontweight='bold')
axes[2].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()

fig.savefig('holt_winters_components.pdf', bbox_inches='tight', transparent=True, dpi=300, facecolor='none', edgecolor='none')
fig.savefig('holt_winters_components.png', bbox_inches='tight', transparent=True, dpi=300, facecolor='none', edgecolor='none')
print("Saved: holt_winters_components.pdf")
print("Saved: holt_winters_components.png")
plt.show()

## 7. Key Takeaways

### Why Holt-Winters Captures Seasonality Better

1. **Simple ES** produces a flat forecast - it only learns the overall level and cannot adapt to seasonal patterns.

2. **Holt's method** captures trend but still produces straight-line forecasts that miss the cyclical nature of energy prices.

3. **Holt-Winters** explicitly models the seasonal component, allowing forecasts to follow the expected annual pattern.

### Practical Applications in Energy Markets

- **Natural Gas**: Winter heating demand creates predictable price spikes
- **Electricity**: Summer cooling loads and winter heating affect prices seasonally
- **Renewable Energy**: Solar and wind generation follow seasonal patterns

### Model Selection Guidelines

| Data Characteristics | Recommended Model |
|---------------------|------------------|
| No trend, no seasonality | Simple ES |
| Trend only | Holt |
| Trend + Seasonality | Holt-Winters |

### Limitations

- Exponential smoothing assumes **stable patterns** - sudden structural breaks can degrade forecasts
- The seasonal period must be known in advance (requires domain knowledge)
- For volatile commodities like natural gas, consider combining with other methods (e.g., GARCH for volatility)